# Config

In [1]:
!pip install nltk==3.9.1
!pip install mlflow==3.3.1

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 14.8 MB/s  0:00:00
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 15.1 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 15.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 15.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 703.4/703.4 kB 15.0 MB/s  0:00:00
  Attempting uninstall: werkzeug
    Found existing installation: Werkzeug 3.0.0
    Uninstalling Werkzeug-3.0.0:━━━━━━━━━━━━━━━━  0/26 [werkzeug]
      Successfully uninstalled Werkzeug-3.0.0━━━  0/26 [werkzeug]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26/26 [mlflow]25/26 [mlflow]skinny]]pi]


In [ ]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [3]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json

# 1) Preprocesamiento de los datos


In [6]:
# 1) Cargar datos
path = "/tmp/data"
path_analytics = "/tmp/analytics"
filePATH = os.path.join(path, "data_concatenada.xlsx")
df = pd.read_excel(filePATH,
                   usecols=["Código VRID", "Título", "Resumen", "Keywords", "Interdisciplinario", "Transdisciplinario", "Facultad del Proyecto",
                            "Depto Persona"]) \
       .fillna("")

# 2) Guardar qué secuencias de palabras del resumen serán eliminadas al aplicar get_expressions_to_delete()
list_texts = df["Resumen"].to_list()
df_deleted = check_deleted_expressions(list_texts)
savepath=os.path.join(path_analytics, "deleted_re.xlsx")
df_deleted.to_excel(savepath, index=False)

# 3) Preprocesar los datos
#Columnas que se van a preprocesar
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Título": "Titulo_trad",
    "Resumen": "Resumen_trad",
    "Keywords": "keywords_trad",
    "Facultad del Proyecto": "Facultad_del_Proyecto_trad",
    "Depto Persona": "Depto_Persona_trad",
}
#Preprocesamiento de datos
df[list(cols.values())] = df[list(cols.keys())].applymap(clean_text)
savepath=os.path.join(path, "data_clean.xlsx")
df.to_excel(savepath, index=False)

# 2) Traducción del texto

In [ ]:
#Crear columna de registro de idioma: 
# True: Texto en español, False: Texto en inglés
df["Español"]=detect_language(df["Resumen_trad"])

In [9]:
from transformers import MarianMTModel, MarianTokenizer

#1. Cargar modelo de traducción
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
trans = translator(model, tokenizer)

#Columnas que se van a traducir
#Nombre fila seleccionada:Columna que se creará para guardar resultado
cols = {
    "Titulo_trad": "Titulo_trad",
    "Resumen_trad": "Resumen_trad",
    "keywords_trad": "keywords_trad",
    "Facultad_del_Proyecto_trad": "Facultad_del_Proyecto_trad",
    "Depto_Persona_trad": "Depto_Persona_trad",
}

#2. Traducción de columnas
#####Estoy trabajando en mejorar esta parte para que sea más rápida con paralelización por batches
start = time.time()
for src, dst in cols.items():
    df[dst] = trans.translate_parallel(df[src].to_list(), batch_size=8)
end = time.time()


#3.Guardado de resultados
savepath=os.path.join(path, "data_translated.xlsx")
df.to_excel(savepath, index=False)

print(f"Tiempo total de traducción: {end - start:.2f} segundos")

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Usando dispositivo: cuda


model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Traduciendo: 100%|██████████| 119/119 [00:25<00:00,  4.64batch/s]


Tiempo total de traducción: 755.99 segundos


In [10]:
#3. Selección de columnas que se utilizarán en clasificador y concatenación
# Última limpieza antes de generar concatenación
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad", "Facultad_del_Proyecto_trad", "Depto_Persona_trad"]
for col in cols:
    df[col] = df[col].apply(final_clean)

#  Selección de columnas para embedding.
cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]
element_names = ["title", "keywords", "abstract"]
df = gen_text_for_embedding(df, cols, element_names)

# Guardado de resultados
savepath=os.path.join(path, "data_translated_concat.xlsx")
df.to_excel(savepath, index=False)
savepath=os.path.join(path, "data_translated_concat.csv")
df.to_csv(savepath, index=False, encoding="utf-8-sig")

df.head()

,Código VRID,Interdisciplinario,Transdisciplinario,Título,Keywords,Resumen,Facultad del Proyecto,Depto Persona,Titulo_trad,Resumen_trad,keywords_trad,Facultad_del_Proyecto_trad,Depto_Persona_trad,Español,text_for_embedding_translated
0,217.173.049-1.0,SI,,PATRONES DE CRIANZA Y SOCIALIZACIÓN DE GÉNERO ...,,OBJETIVOS GENERALES: _x000D_\nDESCRIBIR LOS PR...,FACULTAD DE CIENCIAS SOCIALES,"DEPARTAMENTO DE ECONOMÍA, SIN INFORMACIÓN, ESC...",patterns of gender upbringing and socializatio...,general objectives: to describe the processes ...,,faculty of social sciences,"department of economics, without information, ...",True,title: patterns of gender upbringing and socia...
1,218.201.002-1.0,SI,,ADAPTACIÓN CULTURAL Y VALIDACIÓN DE LA ESCALA ...,"ESTILO DE VIDA, ADOLESCENTES _x000D_\n",PARA EVALUAR LOS COMPORTAMIENTOS RELACIONADOS ...,FACULTAD DE ENFERMERÍA,"DEPARTAMENTO DE CIENCIA ANIMAL, DEPARTAMENTO D...",cultural adaptation and validation of the life...,in order to evaluate the behaviors related to ...,"lifestyle, teens",faculty of nursing,"department of animal science, department of pl...",True,title: cultural adaptation and validation of t...
2,218.102.031-1.0IN,NO,,PROMOVIENDO LA REFLEXIÓN EN ESTUDIANTES DE PRE...,,EL PRESENTE PROYECTO INVOLUCRA LA REALIZACIÓN ...,FACULTAD DE ODONTOLOGÍA,DEPARTAMENTO DE ASTRONOMÍA,promoting reflection in preclinical dental stu...,the present project involves the realization o...,,faculty of dentistry,department of astronomy,True,title: promoting reflection in preclinical den...
3,218.163.016-INI,INDEFINIDO,,MOTIVACIÓN Y HABILIDADES SOCIALES EN ADOLESCENTES,,EL ESTUDIO DE LA MOTIVACIÓN TIENE DIFERENTES A...,FACULTAD DE EDUCACIÓN,"DEPARTAMENTO DE CIENCIAS DE LA EDUCACIÓN, DEPT...",motivation and social skills in adolescents,the study of the motivation has different side...,,faculty of education,"department of education sciences, department o...",True,title: motivation and social skills in adolesc...
4,219.091.052-INI,NO,,TIME EFFECTS ON THE LIQUEFACTION RESPONSE OF G...,,SECONDARY CONSOLIDATION AND AGEING ARE TWO OFT...,FACULTAD DE INGENIERÍA,DEPTO. TEORÍA POLITICA Y FUND.DE LA EDUC.,time effects on the liquefaction response of g...,secondary consolidation and ageing are two oft...,,faculty of engineering,department of political and fund theory of edu...,False,title: time effects on the liquefaction respon...


# 3) Split dataset

In [11]:
def to_serializable(obj):
    if hasattr(obj, "tolist"):
        return obj.tolist()
    return obj

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
import json

path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

df = df[df["Interdisciplinario"] != "INDEFINIDO"]
le = LabelEncoder()
df["labels"] = le.fit_transform(df["Interdisciplinario"])

# Convertir a arrays
ids = df["Código VRID"].to_numpy()
labels = df["labels"].to_numpy()

# Train/Test split (ids y labels en paralelo)
idx_train, idx_test, y_train, y_test = train_test_split(
    ids,
    labels,
    test_size=0.2,
    random_state=7,
    stratify=labels
)

# Crear folds sobre train
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)

folds = []
for fold, (train_pos, val_pos) in enumerate(skf.split(idx_train, y_train)):
    train_ids = idx_train[train_pos]   # array de IDs
    val_ids = idx_train[val_pos]       # array de IDs
    folds.append(val_ids)

print("Test size:", len(idx_test))
print("Fold 0 - Val size:", len(folds[0]))

#Guardar index en diccionario
dataset_index = {
    "Train": idx_train,
    "Test": idx_test,
    "kfolds": folds 
}
filepath=os.path.join(path, "train_test_ids_3folds.json")

# Guardar
with open(filepath, "w", encoding="utf-8") as f:
    json.dump(dataset_index, f, default=to_serializable, indent=2, ensure_ascii=False)


Test size: 193
Fold 0 - Val size: 257


# 4) TF-ID feature extractor 

## Train

In [31]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

#Prueba con solo textos traducidos
#df = df[df["Español"]==False]

In [32]:
from sklearn.preprocessing import LabelEncoder
from models.TIFD import gen_TFID_dataset, preprocess_text_for_TFID, gen_TFID_vectors
import numpy as np

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_TFID_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_TFID_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

#Lemantización y eliminación de stopwords
X_train = preprocess_text_for_TFID(X_train)
# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)

In [ ]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.cv import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=2
sample_weight_On=True
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

In [27]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

{'accuracy': 0.7948717948717948, 'precision': 0.7777777777777778, 'recall': 0.7777777777777778, 'f1_score': 0.7948717948717948, 'cm': array([[34,  8],
       [ 8, 28]]), 'f1_es': 0.0, 'f1_en': 0.7948717948717948, 'cm_es': array([], shape=(0, 0), dtype=int64), 'cm_en': array([[34,  8],
       [ 8, 28]])}
{'accuracy': 0.6923076923076923, 'precision': 0.6363636363636364, 'recall': 0.7777777777777778, 'f1_score': 0.691497975708502, 'cm': array([[26, 16],
       [ 8, 28]]), 'f1_es': 0.0, 'f1_en': 0.691497975708502, 'cm_es': array([], shape=(0, 0), dtype=int64), 'cm_en': array([[26, 16],
       [ 8, 28]])}
{'accuracy': 0.5256410256410257, 'precision': 0.48936170212765956, 'recall': 0.6388888888888888, 'f1_score': 0.5213350768722942, 'cm': array([[18, 24],
       [13, 23]]), 'f1_es': 0.0, 'f1_en': 0.5213350768722942, 'cm_es': array([], shape=(0, 0), dtype=int64), 'cm_en': array([[18, 24],
       [13, 23]])}
{'accuracy': 0.7435897435897436, 'precision': 0.75, 'recall': 0.6666666666666666, 'f1_

## Save

In [ ]:
import mlflow
import git 
#MLflow logging helper function
def safe_log_metric(name, value):
    try:
        if isinstance(value, (list, tuple, np.ndarray)):
            if np.size(value) == 1:
                value = float(np.array(value).item())
            else:
                raise ValueError("Métrica con más de un valor.")
        else:
            value = float(value)
        mlflow.log_metric(name, value)
    except Exception as e:
        print(f"⚠️ No se pudo loggear {name}: {e}")

def mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es):
    # Set backend store
    mlflow.set_tracking_uri("http://mlflow-server:5000")
    tracking_uri = mlflow.get_tracking_uri()
    print("Current tracking uri: {}".format(tracking_uri)) 

    # Define el experimento (lo crea si no existe)
    mlflow.set_experiment(exp_info["exp_name"])
    
    # Obtener commit actual
    repo = git.Repo(search_parent_directories=True)
    commit_hash = repo.head.object.hexsha

    for model_name, metrics in results_val.items():
        model = models_dicc[model_name]

        with mlflow.start_run(run_name=model_name):
            print(f"📝 Registrando modelo en MLflow: {model_name}")

            # Hiperparámetros
            try:
                mlflow.log_params(model.get_params())
            except:
                print(f"⚠️ No se pudieron loggear los hiperparámetros para {model_name}")

            #Parámetros adicionales
            for k, v in extra_parms.items():
                mlflow.log_param(k, v)

            # Métricas de validación
            for k, v in metrics.items():
                safe_log_metric(f"val_{k}", v)

            # Métricas de test
            results_test = eval_model(model, X_test, y_test, lang_es)
            for k, v in results_test.items():
                if k.startswith("cm"):
                    # Guardar confusion matrix (o similar) como artefacto
                    # Guardar como CSV temporal
                    fname = f"{k}.csv"
                    np.savetxt(fname, v, delimiter=",", fmt="%d")

                    mlflow.log_artifact(fname, artifact_path="confusion_matrices")

                    # Eliminar archivo local si no lo necesitas
                    os.remove(fname)

                else:
                    # Guardar métrica numérica
                    safe_log_metric(f"test_{k}", v)
            
            #Guardar commit de git
            mlflow.log_param("git_commit", commit_hash)
                    
            # Guardar modelo
            mlflow.sklearn.log_model(model, name = "model", input_example=X_test[:5])

In [ ]:
from pipelines.ML_pipeline_skp import mlflow_ckeckpoint

exp_info = {
    'exp_name': "Bayesiansearchcv_TFID",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

2025/08/25 20:26:28 INFO mlflow.tracking.fluent: Experiment with name 'Bayesiansearchcv_TFID_EN' does not exist. Creating a new experiment.


Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression
🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/413483937166636898/runs/cb3a0f114739408c96ef5ae0c4402aef
🧪 View experiment at: http://mlflow-server:5000/#/experiments/413483937166636898
📝 Registrando modelo en MLflow: RandomForestClassifier
🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/413483937166636898/runs/ba2e8c689bbe496bb895a36885f42358
🧪 View experiment at: http://mlflow-server:5000/#/experiments/413483937166636898
📝 Registrando modelo en MLflow: XGBClassifier
🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/413483937166636898/runs/13cb880a4e10403b87a881fa77f18589
🧪 View experiment at: http://mlflow-server:5000/#/experiments/413483937166636898
📝 Registrando modelo en MLflow: SVC
🏃 View run SVC at: http://mlflow-server:5000/#/experiments/413483937166636898/runs/9b33ed8603f04d52a49aa5c32212df47
🧪 View experi

# 5) SPECTER model

In [63]:
import torch
from transformers import BertModel, BertTokenizer
import numpy as np
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

def embed_texts(texts, BASE_MODEL, ADAPTER_NAME, batch_size=32, device='cpu'):
    #Parameters
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    model = AutoAdapterModel.from_pretrained(BASE_MODEL, trust_remote_code=True)
    model.load_adapter(ADAPTER_NAME, source="hf", set_active=True)
    model.to(device)
    model.eval()
    print("Modelo SPECTER2 cargado correctamente.")
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch,
                            padding=True,
                            truncation=True,
                            return_tensors="pt",
                            max_length=512).to(device)
        with torch.no_grad():
            out = model(**encoded)
            embs = out.last_hidden_state[:, 0, :]
        embs = torch.nn.functional.normalize(embs, p=2, dim=1)
        embeddings.append(embs.cpu().numpy())
    return np.vstack(embeddings)

In [ ]:
def gen_dataset(codes_vrid, df):
    #Selección unicamente de elementos de df que se encuentren en codes_vrid
    df = df[df["Código VRID"].isin(codes_vrid)].copy()

    #Creación de index en función de orden de los datos
    df['idx'] = np.arange(0, df.shape[0])

    #Generación de datasets
    X = df["text_for_embedding_translated"].to_list()
    y = df["Interdisciplinario"].to_list()
    return X, y, df

In [4]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [ ]:
#del gen_dataset
#from utils.dataset import gen_dataset

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# 2) Calcular embeddings
# Parámetros modelo
BASE_MODEL = "allenai/specter2_base"
ADAPTER_NAME = "allenai/specter2"
X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)

print(X_train.shape, X_test.shape)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


In [69]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.cv import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [1.0, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbf

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.63, 'std_test_score': 0.02}
RandomForestClassifier: {'mean_test_score': 0.64, 'std_test_score': 0.01}
XGBClassifier: {'mean_test_score': 0.65, 'std_test_score': 0.01}
SVC: {'mean_test_score': 0.65, 'std_test_score': 0.03}


In [71]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

{'accuracy': 0.6321243523316062, 'precision': 0.7, 'recall': 0.6306306306306306, 'f1_score': 0.6340969827510602, 'cm': array([[52, 30],
       [41, 70]]), 'f1_es': 0.607514752138559, 'f1_en': 0.6541666666666668, 'cm_es': array([[17, 23],
       [22, 53]]), 'cm_en': array([[35,  7],
       [19, 17]])}
{'accuracy': 0.6839378238341969, 'precision': 0.7232142857142857, 'recall': 0.7297297297297297, 'f1_score': 0.6836769675442034, 'cm': array([[51, 31],
       [30, 81]]), 'f1_es': 0.6550350631136045, 'f1_en': 0.7160633484162896, 'cm_es': array([[18, 22],
       [17, 58]]), 'cm_en': array([[33,  9],
       [13, 23]])}
{'accuracy': 0.6424870466321243, 'precision': 0.6810344827586207, 'recall': 0.7117117117117117, 'f1_score': 0.6408031411082682, 'cm': array([[45, 37],
       [32, 79]]), 'f1_es': 0.6269999096086053, 'f1_en': 0.653444649799248, 'cm_es': array([[16, 24],
       [18, 57]]), 'cm_en': array([[29, 13],
       [14, 22]])}
{'accuracy': 0.6683937823834197, 'precision': 0.711711711711711